In [42]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

In [43]:
CURRENT_FILES_mA = {5: "5mA.csv", 10: "10mA.csv", 15: "15mA.csv"}

S_cm = 0.200              # probe spacing, cm
W_cm = 0.5 * 0.1          # mm -> cm
W_over_S = W_cm / S_cm

# --- Literature value for comparison -------------------------
Eg_LITERATURE_eV = 0.67   # standard band gap of Ge

# --- Physical constants ---------------------------------------
kB_eV_per_K = 8.617333262e-5   # Boltzmann constant, eV/K

dV_mV = 0.1                # +/- voltmeter reading resolution, mV

# --- Intrinsic-region auto-detection ---------------------------
MIN_INTRINSIC_POINTS = 6
R2_THRESHOLD = 0.999        # "good enough" linearity to keep extending the window

# ============================================================
# Thin-slice correction factor G7(W/S)
# ============================================================
# Table 1 (tabulated values, for reference / sanity-check only):
TABLE_WS = np.array([0.100, 0.141, 0.200, 0.330, 0.500, 1.000,
                      1.414, 2.000, 3.333, 5.000, 10.000])
TABLE_G7 = np.array([13.863, 9.704, 6.931, 4.159, 2.780, 1.504,
                      1.223, 1.094, 1.0228, 1.0070, 1.00045])

In [44]:
def G7_thin_slice_asymptote(ws):
    """
    Correction divisor G7(W/S) for a four-probe measurement on a thin
    slice with a non-conducting bottom boundary, in the thin-slice
    limit W/S << 1:
                 G7(W/S) -> (2S/W) * ln(2)      as (W/S) -> 0

    This is the standard asymptotic form of the tabulated/plotted
    correction function (Fig. 2 / Table 1). It reproduces the smallest
    tabulated point, W/S = 0.100 -> G7 = 13.863, essentially exactly:
        (2/0.100)*ln(2) = 13.8629...
    Our sample has W/S = 0.025, below the tabulated range (min 0.100),
    so we use this asymptotic formula rather than extrapolating the
    table by eye.
    """
    return (2.0 / ws) * np.log(2.0)


def get_G7(ws):
    if ws < TABLE_WS.min():
        return G7_thin_slice_asymptote(ws), "asymptotic formula (2S/W)ln2 [W/S below tabulated range]"
    else:
        # simple log-log interpolation against the tabulated curve
        val = np.interp(np.log(ws), np.log(TABLE_WS), np.log(TABLE_G7))
        return np.exp(val), "log-log interpolation of Table 1"


G7_VALUE, G7_METHOD = get_G7(W_over_S)


In [45]:
# ============================================================
# Load + process one current file
# ============================================================
def process_file(path, I_mA):
    I_amp = I_mA / 1000.0
    df = pd.read_csv(path)
    df.columns = ["V_thermo_mV", "T0_C", "T_corr_C", "V_heat_mV", "V_cool_mV"]

    T_C = df["T_corr_C"]
    V_avg_mV = (df["V_heat_mV"] + df["V_cool_mV"]) / 2.0
    V_avg_V = V_avg_mV / 1000.0

    rho0 = (V_avg_V / I_amp) * 2 * np.pi * S_cm      # ohm.cm, uncorrected
    rho = rho0 / G7_VALUE                             # ohm.cm, thin-slice corrected

    T_K = T_C + 273.15
    inv_T_1000 = 1000.0 / T_K

    out = pd.DataFrame({
        "S_No": np.arange(1, len(df) + 1),
        "I_mA": I_mA,
        "T0_raw_C": df["T0_C"],
        "T_used_C": T_C,
        "T_K": T_K,
        "V_heating_mV": df["V_heat_mV"],
        "V_cooling_mV": df["V_cool_mV"],
        "V_avg_V": V_avg_V,
        "rho0_ohm_cm": rho0,
        "rho_ohm_cm": rho,
        "1000_over_T": inv_T_1000,
        "log10_rho": np.log10(rho),
        "ln_rho": np.log(rho),
    })
    return out.sort_values("T_K").reset_index(drop=True)



In [46]:
# ============================================================
# Auto-detect the linear ("intrinsic") high-T region
# ============================================================
def find_intrinsic_window(x_1000_over_T, y_log_rho,
                           min_points=MIN_INTRINSIC_POINTS,
                           r2_threshold=R2_THRESHOLD):
    """
    The intrinsic region is the HIGH-temperature (i.e. SMALL 1000/T)
    end of the curve. Anchor a window at the smallest-x point and grow
    it (toward larger x / lower T) as long as the linear fit R^2 stays
    above `r2_threshold`; return the largest such window. If none
    reaches the threshold, fall back to whichever window (of at least
    `min_points`) has the best R^2.
    """
    order = np.argsort(x_1000_over_T)
    xs = np.asarray(x_1000_over_T)[order]
    ys = np.asarray(y_log_rho)[order]
    n = len(xs)

    best_meeting_threshold = None
    best_overall = None
    for end in range(min_points, n + 1):
        xx, yy = xs[:end], ys[:end]
        res = stats.linregress(xx, yy)
        r2 = res.rvalue ** 2
        if best_overall is None or r2 > best_overall[-1]:
            best_overall = (0, end, res, r2)
        if r2 >= r2_threshold:
            best_meeting_threshold = (0, end, res, r2)

    chosen = best_meeting_threshold if best_meeting_threshold is not None else best_overall
    start_idx, end_idx, res, r2 = chosen
    mask_sorted = np.zeros(n, dtype=bool)
    mask_sorted[start_idx:end_idx] = True
    mask = np.zeros(n, dtype=bool)
    mask[order[start_idx:end_idx]] = True
    return mask, res, r2


def eg_from_fit(slope, slope_stderr, base):
    """
    base = 'log10' or 'ln'.
    Log10(rho) = [Eg/(2 kB T)]/ln(10) - log10(K)   => slope (in 1000/T units) = Eg/(2000 kB)/ln(10)
    Ln(rho)    = [Eg/(2 kB T)]        - ln(K)       => slope (in 1000/T units) = Eg/(2000 kB)
    """
    if base == "log10":
        factor = 2000.0 * kB_eV_per_K * np.log(10.0)   # ~0.3970
    elif base == "ln":
        factor = 2000.0 * kB_eV_per_K                    # ~0.1723
    else:
        raise ValueError(base)
    Eg = slope * factor
    dEg = abs(slope_stderr * factor)
    return Eg, dEg, factor

In [47]:
# ============================================================
# MAIN
# ============================================================
def main():
    per_current = {}
    for I_mA, fname in CURRENT_FILES_mA.items():
        path = os.path.join('./', fname)
        df = process_file(path, I_mA)
        per_current[I_mA] = df
        df.to_csv(os.path.join("./", f"processed_{I_mA}mA.csv"), index=False)

    combined = pd.concat(per_current.values(), ignore_index=True).sort_values("T_K").reset_index(drop=True)
    combined.to_csv(os.path.join("./", "processed_combined.csv"), index=False)

    # ---- auto-detect intrinsic region on the COMBINED data ----
    mask, fit_log10, r2_log10 = find_intrinsic_window(
        combined["1000_over_T"].values, combined["log10_rho"].values)
    T_lo = combined.loc[mask, "T_K"].min()
    T_hi = combined.loc[mask, "T_K"].max()
    n_intrinsic = mask.sum()

    Eg_combined, dEg_combined, factor10 = eg_from_fit(fit_log10.slope, fit_log10.stderr, "log10")

    # ---- same window, fit with natural log (should agree exactly) ----
    fit_ln = stats.linregress(combined.loc[mask, "1000_over_T"], combined.loc[mask, "ln_rho"])
    Eg_combined_ln, dEg_combined_ln, factor_ln = eg_from_fit(fit_ln.slope, fit_ln.stderr, "ln")

    # ---- per-current fits over the SAME temperature window (reproducibility check) ----
    per_current_fit = {}
    for I_mA, df in per_current.items():
        sub_mask = (df["T_K"] >= T_lo) & (df["T_K"] <= T_hi)
        if sub_mask.sum() < 3:
            continue
        res = stats.linregress(df.loc[sub_mask, "1000_over_T"], df.loc[sub_mask, "log10_rho"])
        Eg_i, dEg_i, _ = eg_from_fit(res.slope, res.stderr, "log10")
        per_current_fit[I_mA] = dict(slope=res.slope, stderr=res.stderr, r2=res.rvalue**2,
                                      n=int(sub_mask.sum()), Eg=Eg_i, dEg=dEg_i)

    Eg_values = np.array([v["Eg"] for v in per_current_fit.values()])
    Eg_repro_mean = Eg_values.mean()
    Eg_repro_std = Eg_values.std(ddof=1) if len(Eg_values) > 1 else 0.0

    pct_error = 100.0 * abs(Eg_combined - Eg_LITERATURE_eV) / Eg_LITERATURE_eV

    # ---- heating vs cooling hysteresis check (uses raw voltages) ----
    hyst_rows = []
    for I_mA, fname in CURRENT_FILES_mA.items():
        raw = pd.read_csv(os.path.join("./", fname))
        raw.columns = ["V_thermo_mV", "T0_C", "T_corr_C", "V_heat_mV", "V_cool_mV"]
        frac_diff = (raw["V_heat_mV"] - raw["V_cool_mV"]).abs() / ((raw["V_heat_mV"] + raw["V_cool_mV"]) / 2.0)
        hyst_rows.append((I_mA, frac_diff.mean() * 100, frac_diff.max() * 100))

    # ============================================================
    # PLOTS
    # ============================================================
    colors = {5: "tab:blue", 10: "tab:orange", 15: "tab:green"}
    markers = {5: "o", 10: "s", 15: "^"}

    # --- Required plot 1: Log10(rho) vs 1000/T ---
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for I_mA, df in per_current.items():
        ax.scatter(df["1000_over_T"], df["log10_rho"], s=18, color=colors[I_mA],
                   marker=markers[I_mA], label=f"{I_mA} mA", alpha=0.8)
    xx = np.array([combined.loc[mask, "1000_over_T"].min(), combined.loc[mask, "1000_over_T"].max()])
    ax.plot(xx, fit_log10.intercept + fit_log10.slope * xx, "k--", lw=1.8,
            label=f"intrinsic-region fit (R$^2$={r2_log10:.4f})")
    ax.axvspan(xx[0], xx[1], color="grey", alpha=0.08, label="fitted (intrinsic) region")
    ax.set_xlabel(r"$1000/T$  (K$^{-1}$)")
    ax.set_ylabel(r"$\log_{10}\rho$  ($\rho$ in $\Omega\cdot$cm)")
    ax.set_title("Ge four-probe resistivity: $\\log_{10}\\rho$ vs $1000/T$")
    txt = (f"slope = {fit_log10.slope:.4f} $\\pm$ {fit_log10.stderr:.4f}\n"
           f"$E_g$ = {Eg_combined:.4f} $\\pm$ {dEg_combined:.4f} eV\n"
           f"(lit. $E_g$ = {Eg_LITERATURE_eV} eV, {pct_error:.1f}% off)")
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, va="top", ha="left",
            fontsize=9, bbox=dict(boxstyle="round", fc="white", ec="0.5", alpha=0.9))
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join("./", "log10_rho_vs_1000_T.png"), dpi=200)
    plt.close(fig)

    # --- Required plot 2: Loge(rho) vs 1000/T ---
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for I_mA, df in per_current.items():
        ax.scatter(df["1000_over_T"], df["ln_rho"], s=18, color=colors[I_mA],
                   marker=markers[I_mA], label=f"{I_mA} mA", alpha=0.8)
    ax.plot(xx, fit_ln.intercept + fit_ln.slope * xx, "k--", lw=1.8,
            label=f"intrinsic-region fit (R$^2$={fit_ln.rvalue**2:.4f})")
    ax.axvspan(xx[0], xx[1], color="grey", alpha=0.08, label="fitted (intrinsic) region")
    ax.set_xlabel(r"$1000/T$  (K$^{-1}$)")
    ax.set_ylabel(r"$\ln\rho$  ($\rho$ in $\Omega\cdot$cm)")
    ax.set_title("Ge four-probe resistivity: $\\ln\\rho$ vs $1000/T$")
    txt = (f"slope = {fit_ln.slope:.4f} $\\pm$ {fit_ln.stderr:.4f}\n"
           f"$E_g$ = {Eg_combined_ln:.4f} $\\pm$ {dEg_combined_ln:.4f} eV")
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, va="top", ha="left",
            fontsize=9, bbox=dict(boxstyle="round", fc="white", ec="0.5", alpha=0.9))
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join("./", "ln_rho_vs_1000_T.png"), dpi=200)
    plt.close(fig)

    # --- Bonus plot: rho vs T(K), linear axes ---
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for I_mA, df in per_current.items():
        ax.plot(df["T_K"], df["rho_ohm_cm"], color=colors[I_mA], marker=markers[I_mA],
                ms=4, lw=1, label=f"{I_mA} mA")
    ax.axvspan(T_lo, T_hi, color="grey", alpha=0.12, label="intrinsic region")
    ax.set_xlabel("T (K)")
    ax.set_ylabel(r"$\rho$  ($\Omega\cdot$cm)")
    ax.set_title("Ge resistivity vs temperature")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join("./", "rho_vs_T.png"), dpi=200)
    plt.close(fig)

    # --- Bonus plot: heating vs cooling repeatability ---
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=False)
    for ax, (I_mA, fname) in zip(axes, CURRENT_FILES_mA.items()):
        raw = pd.read_csv(os.path.join("./", fname))
        raw.columns = ["V_thermo_mV", "T0_C", "T_corr_C", "V_heat_mV", "V_cool_mV"]
        ax.plot(raw["T_corr_C"], raw["V_heat_mV"], "o-", ms=3, label="heating", color="crimson")
        ax.plot(raw["T_corr_C"], raw["V_cool_mV"], "s-", ms=3, label="cooling", color="navy")
        ax.set_title(f"{I_mA} mA")
        ax.set_xlabel("T (deg C)")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("4-probe voltage (mV)")
    axes[0].legend(fontsize=8)
    fig.suptitle("Heating vs. cooling repeatability check")
    fig.tight_layout()
    fig.savefig(os.path.join("./", "heating_cooling_check.png"), dpi=200)
    plt.close(fig)

    # ============================================================
    # RESULTS SUMMARY
    # ============================================================
    lines = []
    lines.append("=" * 70)
    lines.append("GE FOUR-PROBE BAND GAP ANALYSIS -- RESULTS SUMMARY")
    lines.append("=" * 70)
    lines.append("")
    lines.append("Geometry / correction factor")
    lines.append("-" * 70)
    lines.append(f"  S (probe spacing)          = {S_cm:.3f} cm")
    lines.append(f"  W (crystal thickness)      = {W_cm:.4f} cm  (0.05 mm)")
    lines.append(f"  W/S                        = {W_over_S:.4f}")
    lines.append(f"  G7(W/S)                    = {G7_VALUE:.4f}   [{G7_METHOD}]")
    lines.append("")

    lines.append("Auto-detected intrinsic (linear) region")
    lines.append("-" * 70)
    lines.append(f"  Temperature range          = {T_lo:.1f} K  to  {T_hi:.1f} K "
                 f"({T_lo-273.15:.1f} to {T_hi-273.15:.1f} deg C)")
    lines.append(f"  Number of points used      = {n_intrinsic}  (out of {len(combined)} combined)")
    lines.append(f"  R^2 of fit                 = {r2_log10:.5f}")
    lines.append("")

    lines.append("Band gap from combined (all-current) fit")
    lines.append("-" * 70)
    lines.append(f"  Log10(rho) vs 1000/T slope = {fit_log10.slope:.5f} +/- {fit_log10.stderr:.5f}")
    lines.append(f"  ==> E_g (from log10 fit)   = {Eg_combined:.4f} +/- {dEg_combined:.4f} eV")
    lines.append(f"  Ln(rho)   vs 1000/T slope  = {fit_ln.slope:.5f} +/- {fit_ln.stderr:.5f}")
    lines.append(f"  ==> E_g (from ln fit)      = {Eg_combined_ln:.4f} +/- {dEg_combined_ln:.4f} eV")
    lines.append(f"  (both bases must and do agree, since ln(rho)=ln(10)*log10(rho))")
    lines.append("")
    lines.append(f"  Literature E_g (Ge)        = {Eg_LITERATURE_eV} eV")
    lines.append(f"  Deviation from literature  = {pct_error:.2f} %")
    lines.append("")

    lines.append("Reproducibility across the three independent current settings")
    lines.append("(same intrinsic T-window fitted separately per current)")
    lines.append("-" * 70)
    for I_mA, v in per_current_fit.items():
        lines.append(f"  {I_mA:>2d} mA: slope={v['slope']:.5f}+/-{v['stderr']:.5f}  "
                     f"R^2={v['r2']:.5f}  n={v['n']:>2d}   E_g={v['Eg']:.4f}+/-{v['dEg']:.4f} eV")
    lines.append(f"  Mean across currents        = {Eg_repro_mean:.4f} eV")
    lines.append(f"  Std-dev across currents     = {Eg_repro_std:.4f} eV  "
                 f"(empirical run-to-run/reproducibility uncertainty)")
    lines.append("")

    lines.append("Heating vs. cooling hysteresis (fractional |V_heat-V_cool|/V_avg)")
    lines.append("-" * 70)
    for I_mA, mean_pct, max_pct in hyst_rows:
        lines.append(f"  {I_mA:>2d} mA: mean = {mean_pct:.2f} %   max = {max_pct:.2f} %")
    lines.append("")

    lines.append("Error-analysis notes")
    lines.append("-" * 70)
    lines.append("  * The dominant, reported uncertainty on E_g is the STANDARD ERROR")
    lines.append("    OF THE SLOPE from the least-squares fit to log10(rho) vs 1000/T,")
    lines.append("    propagated via E_g = slope * 2000*kB*ln(10):")
    lines.append(f"        dE_g = |d(slope)| * {factor10:.4f} eV = {dEg_combined:.4f} eV")
    lines.append("  * Uncertainties in S, I, and G7(W/S) are SYSTEMATIC: they rescale")
    lines.append("    every rho(T) point by the same factor, which shifts the fitted")
    lines.append("    line vertically (changes the prefactor K) but leaves its SLOPE,")
    lines.append("    and hence E_g, unchanged to first order. They are therefore not")
    lines.append("    included in the quoted dE_g, but are listed below for reference:")
    lines.append(f"      assumed dV = +/-{dV_mV} mV")
    lines.append("  * The spread of E_g across the three independently-run currents")
    lines.append(f"    (std dev = {Eg_repro_std:.4f} eV) is an independent, empirical")
    lines.append("    check of reproducibility and captures effects (contact")
    lines.append("    resistance drift, thermocouple placement, etc.) that a single")
    lines.append("    regression's standard error cannot see.")
    lines.append("=" * 70)

    summary_text = "\n".join(lines)
    with open("results_summary.txt", "w") as f:
        f.write(summary_text)

    print(summary_text)

In [48]:

if __name__ == "__main__":
    main()

GE FOUR-PROBE BAND GAP ANALYSIS -- RESULTS SUMMARY

Geometry / correction factor
----------------------------------------------------------------------
  S (probe spacing)          = 0.200 cm
  W (crystal thickness)      = 0.0500 cm  (0.05 mm)
  W/S                        = 0.2500
  G7(W/S)                    = 5.5203   [log-log interpolation of Table 1]

Auto-detected intrinsic (linear) region
----------------------------------------------------------------------
  Temperature range          = 338.6 K  to  449.1 K (65.5 to 176.0 deg C)
  Number of points used      = 172  (out of 181 combined)
  R^2 of fit                 = 0.99883

Band gap from combined (all-current) fit
----------------------------------------------------------------------
  Log10(rho) vs 1000/T slope = 1.79543 +/- 0.00471
  ==> E_g (from log10 fit)   = 0.7125 +/- 0.0019 eV
  Ln(rho)   vs 1000/T slope  = 4.13412 +/- 0.01083
  ==> E_g (from ln fit)      = 0.7125 +/- 0.0019 eV
  (both bases must and do agree, since ln